# FTMO AI Trading System — Cloud Inference
**Runs LightGBM + CatBoost inference on all 8 pretrained symbol pairs.**

### Open this notebook directly from GitHub (no Google Drive needed):
```
https://colab.research.google.com/github/Codebreaker-source/AI-Trading-System/blob/main/src/ftmo_system/colab/trading_inference.ipynb
```
Models and code are pulled from GitHub automatically — updates deploy on next session start.

### One-time setup:
1. Go to **Colab Secrets** (🔑 icon in left sidebar)
2. Add secret named `AZURE_STORAGE_CONNECTION_STRING` — paste your full Azure connection string
3. Toggle **Notebook access** ON for that secret
4. **Runtime → Run all**

The inference loop in the last cell runs forever. Leave this tab open during trading hours.

In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────────────────
!pip install -q azure-storage-blob lightgbm catboost xgboost joblib pandas numpy

In [ ]:
# ── Cell 2: Clone repo using GitHub PAT from Colab Secrets ──────────────
# Requires a secret named GITHUB_TOKEN in the Colab Secrets panel (key icon).
# Create a PAT at: https://github.com/settings/tokens
#   → Generate new token (classic) → scope: repo (read-only is fine)

import os
from google.colab import userdata

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    raise RuntimeError(
        'GITHUB_TOKEN not found in Colab Secrets. '
        'Add it via the key icon in the left sidebar.'
    )

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/Codebreaker-source/AI-Trading-System.git'

if not os.path.exists('/content/AI-Trading-System'):
    !git clone -q "{REPO_URL}" /content/AI-Trading-System
else:
    # Re-authenticate on pull in case token changed
    !git -C /content/AI-Trading-System remote set-url origin "{REPO_URL}"
    !git -C /content/AI-Trading-System pull -q

import sys
sys.path.insert(0, '/content/AI-Trading-System/src/ftmo_system')
sys.path.insert(0, '/content/AI-Trading-System/src/ftmo_system/core')
print('Repo ready.')
!ls /content/AI-Trading-System/models/current/

In [ ]:
# ── Cell 3: Load Azure connection string from Colab Secrets ─────────────
from google.colab import userdata
import os

try:
    AZURE_CONN_STR = userdata.get('AZURE_STORAGE_CONNECTION_STRING')
    os.environ['AZURE_STORAGE_CONNECTION_STRING'] = AZURE_CONN_STR
    print('Azure connection string loaded from Colab Secrets.')
except Exception as e:
    raise RuntimeError(
        'AZURE_STORAGE_CONNECTION_STRING not found in Colab Secrets. '
        'Add it via the key icon in the left sidebar.'
    ) from e

In [ ]:
# ── Cell 4: Load ALL available models dynamically ───────────────────────
# Scans models/current/ for .joblib (LGB/CatBoost) and .pth (Transformer).
# Skips any model whose expected feature count != 27.
# Old models trained on 58/105/108 features are automatically skipped —
# the weekly retrainer will replace them with fresh 27-feature versions.

import joblib, glob, os
import torch
import torch.nn as nn
import numpy as np

MODEL_DIR  = '/content/AI-Trading-System/models/current'
N_FEATURES = 27

# ── PyTorch SimpleTransformer ─────────────────────────────────────────────
class SimpleTransformer(nn.Module):
    def __init__(self, input_dim=27, d_model=64, nhead=4, num_layers=2,
                 num_classes=3, dropout=0.1):
        super().__init__()
        self.embedding  = nn.Linear(input_dim, d_model)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                         dim_feedforward=128, dropout=dropout,
                                         batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.classifier  = nn.Linear(d_model, num_classes)
        self.dropout     = nn.Dropout(dropout)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.embedding(x)
        x = self.transformer(x)
        x = x.squeeze(1)
        x = self.dropout(x)
        return self.classifier(x)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {DEVICE}')


def get_model_feature_count(model):
    """Get feature count, handling sklearn/LightGBM and CatBoost differently."""
    # sklearn / LightGBM standard attribute
    n = getattr(model, 'n_features_in_', None)
    if n and n > 0:
        return n
    n = getattr(model, 'n_features_', None)
    if n and n > 0:
        return n
    # CatBoost: try get_feature_importance (length = feature count)
    if hasattr(model, 'get_feature_importance'):
        try:
            return len(model.get_feature_importance())
        except Exception:
            pass
    # Last resort: test predict with N_FEATURES zeros
    try:
        model.predict_proba(np.zeros((1, N_FEATURES)))
        return N_FEATURES
    except Exception:
        return None

# ── Load sklearn models (.joblib) ─────────────────────────────────────────
lgbm_models     = {}
catboost_models = {}

for path in sorted(glob.glob(f'{MODEL_DIR}/*.joblib')):
    fname = os.path.basename(path)
    try:
        m = joblib.load(path)
        n = get_model_feature_count(m)
        if n != N_FEATURES:
            print(f'  SKIP {fname}: expects {n} features (need {N_FEATURES})')
            continue
        if fname.endswith('_lightgbm.joblib'):
            base = fname[:-len('_lightgbm.joblib')]
            lgbm_models[base] = m
            print(f'  LGB loaded: {base}')
        elif fname.endswith('_catboost.joblib'):
            base = fname[:-len('_catboost.joblib')]
            catboost_models[base] = m
            print(f'  CAT loaded: {base}')
    except Exception as e:
        print(f'  ERROR loading {fname}: {e}')

# ── Load Transformer models (.pth) ────────────────────────────────────────
transformer_models = {}

for path in sorted(glob.glob(f'{MODEL_DIR}/transformer/*.pth')):
    fname      = os.path.basename(path)
    base_clean = fname.replace('_transformer.pth','').replace('.sim','').replace('.i','').upper()
    try:
        state = torch.load(path, map_location=DEVICE)
        if isinstance(state, dict) and 'model_state_dict' in state:
            state = state['model_state_dict']
        input_dim = state['embedding.weight'].shape[1]
        if input_dim != N_FEATURES:
            print(f'  SKIP TRF {base_clean}: trained on {input_dim} features (need {N_FEATURES})')
            continue
        model = SimpleTransformer(input_dim=input_dim).to(DEVICE)
        model.load_state_dict(state)
        model.eval()
        transformer_models[base_clean] = model
        print(f'  TRF loaded: {base_clean}')
    except Exception as e:
        print(f'  ERROR loading TRF {base_clean}: {e}')

print(f'\nLoaded: {len(lgbm_models)} LGB  {len(catboost_models)} CAT  {len(transformer_models)} Transformer')
if len(lgbm_models) == 0 and len(catboost_models) == 0:
    print('NOTE: No models loaded yet. The weekly retrainer (Sunday 08:00) will')
    print('train new 27-feature models and push them here automatically.')

In [ ]:
# ── Cell 5: Feature reader — 27 CLEAN features, no expansion needed ──────
# XGBoost, LightGBM, CatBoost models trained by the weekly retrainer all
# expect 27 raw features (the CLEAN set the EA writes to CSV).
# No feature_expander import required.

import numpy as np
import pandas as pd

N_FEATURES = 27   # exact feature count all models in this system expect

FEAT_NAMES = [
    'close', 'high', 'low', 'volume',
    'sma_20', 'sma_50', 'fast_ema', 'slow_ema',
    'htf_fast_ema', 'htf_slow_ema', 'htf_trend_direction', 'htf_trend_alignment',
    'rsi', 'stoch_k', 'stoch_d', 'momentum',
    'atr', 'bb_upper', 'bb_middle', 'bb_lower', 'volatility',
    'volume_sma', 'volume_ratio', 'price_volume',
    'bullish_sentiment', 'bearish_sentiment', 'net_sentiment',
]

print(f'Feature reader ready — expecting {N_FEATURES} CLEAN features per symbol.')

In [ ]:
# ── Cell 6: Azure helpers ────────────────────────────────────────────────
from azure.storage.blob import BlobServiceClient
import json
from datetime import datetime, timezone
import io

FEATURES_CONTAINER    = 'trading-features'
PREDICTIONS_CONTAINER = 'colab-predictions'

blob_service = BlobServiceClient.from_connection_string(AZURE_CONN_STR)

# Ensure containers exist
for c in [FEATURES_CONTAINER, PREDICTIONS_CONTAINER]:
    try:
        blob_service.create_container(c)
        print(f'Created container: {c}')
    except:
        print(f'Container exists: {c}')

def download_features(symbol: str):
    """Download feature CSV for a symbol from Azure. Returns DataFrame or None."""
    try:
        blob = blob_service.get_blob_client(container=FEATURES_CONTAINER,
                                            blob=f'{symbol}_features.csv')
        raw = blob.download_blob().readall().decode('utf-8')
        df  = pd.read_csv(io.StringIO(raw))
        return df
    except Exception:
        return None

def upload_prediction(symbol: str, source: str, action: str, confidence: float):
    """Upload inference result to Azure."""
    payload = {
        'symbol':     symbol,
        'source':     source,
        'action':     action,
        'confidence': round(float(confidence), 4),
        'timestamp':  datetime.now(timezone.utc).isoformat(),
    }
    blob = blob_service.get_blob_client(container=PREDICTIONS_CONTAINER,
                                        blob=f'{symbol}_{source}_pred.json')
    blob.upload_blob(json.dumps(payload), overwrite=True)

print('Azure helpers ready.')

In [ ]:
# ── Cell 7: Prediction helpers (HOLD-bias fix, all model types) ──────────
# Label encoding: 0=SELL  1=HOLD  2=BUY
# HOLD-bias fix: if HOLD wins but a directional prob ≥ BIAS_THRESHOLD, use it.

BIAS_THRESHOLD = 0.35
LABEL_MAP      = {0: 'SELL', 1: 'HOLD', 2: 'BUY'}


def predict_sklearn(model, features_27: np.ndarray):
    """predict_proba on 27 raw features with HOLD-bias fix."""
    try:
        proba    = model.predict_proba(features_27.reshape(1, -1))[0]
        sell_p, hold_p, buy_p = float(proba[0]), float(proba[1]), float(proba[2])
        pred_idx = int(np.argmax(proba))
        if pred_idx == 1:
            if buy_p  >= BIAS_THRESHOLD and buy_p  > sell_p: return 'BUY',  buy_p
            if sell_p >= BIAS_THRESHOLD and sell_p > buy_p:  return 'SELL', sell_p
            return 'HOLD', hold_p
        return LABEL_MAP[pred_idx], float(proba[pred_idx])
    except Exception as e:
        print(f'sklearn predict error: {e}')
        return 'HOLD', 0.0


def predict_transformer(model, features_27: np.ndarray):
    """SimpleTransformer inference on 27 raw features with HOLD-bias fix."""
    try:
        x = torch.tensor(features_27, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            proba = torch.softmax(model(x), dim=1).cpu().numpy()[0]
        sell_p, hold_p, buy_p = float(proba[0]), float(proba[1]), float(proba[2])
        pred_idx = int(np.argmax(proba))
        if pred_idx == 1:
            if buy_p  >= BIAS_THRESHOLD and buy_p  > sell_p: return 'BUY',  buy_p
            if sell_p >= BIAS_THRESHOLD and sell_p > buy_p:  return 'SELL', sell_p
            return 'HOLD', hold_p
        return LABEL_MAP[pred_idx], float(proba[pred_idx])
    except Exception as e:
        print(f'transformer predict error: {e}')
        return 'HOLD', 0.0


def run_inference_for_symbol(symbol_full: str, base: str):
    """Download feature CSV, read 27 features, run all available models."""
    df = download_features(symbol_full)
    if df is None or df.empty:
        return None, None, None

    latest = df[FEAT_NAMES].iloc[-1].values.astype(np.float32)
    if len(latest) < N_FEATURES:
        return None, None, None
    features_27 = latest[:N_FEATURES]

    lgbm_result = catboost_result = transformer_result = None

    if base in lgbm_models:
        action, conf = predict_sklearn(lgbm_models[base], features_27)
        lgbm_result  = (action, conf)
        if action != 'HOLD':
            upload_prediction(symbol_full, 'lgbm', action, conf)

    if base in catboost_models:
        action, conf    = predict_sklearn(catboost_models[base], features_27)
        catboost_result = (action, conf)
        if action != 'HOLD':
            upload_prediction(symbol_full, 'catboost', action, conf)

    if base in transformer_models:
        action, conf       = predict_transformer(transformer_models[base], features_27)
        transformer_result = (action, conf)
        if action != 'HOLD':
            upload_prediction(symbol_full, 'transformer', action, conf)

    return lgbm_result, catboost_result, transformer_result


print('Inference helpers ready.')

In [ ]:
# ── Cell 8: Dynamic symbol discovery ────────────────────────────────────
# Instead of a hardcoded 8-symbol list, we:
#   1. Scan models/current/ for all *_lightgbm.joblib and *_catboost.joblib files
#   2. Scan Azure trading-features container for symbols with live feature CSVs
#   3. Build SYMBOL_MAP from the intersection — any symbol that has BOTH a model
#      AND fresh features from the EA gets inference run on it automatically.
#
# This means as daily_retrainer.py creates new models and they get pushed to
# GitHub, Colab picks them up on the next session start with no code changes.

import glob
import os

MODEL_DIR = '/content/AI-Trading-System/models/current'

def discover_symbol_map() -> dict:
    """
    Returns { 'EURUSD.sim': 'EURUSD', 'GBPUSD.sim': 'GBPUSD', ... }
    for every base symbol that has at least one model (lgbm or catboost)
    AND has a live feature CSV in Azure.
    """
    # Step 1: find all base symbols with trained models
    model_bases = set()
    for path in glob.glob(f'{MODEL_DIR}/*.joblib'):
        fname = os.path.basename(path)
        for suffix in ('_lightgbm.joblib', '_catboost.joblib', '_xgboost.joblib'):
            if fname.endswith(suffix):
                base = fname[:-len(suffix)]
                model_bases.add(base)

    print(f'Model bases found: {sorted(model_bases)}')

    # Step 2: find all symbols with live feature CSVs in Azure
    live_symbols = set()
    try:
        container = blob_service.get_container_client(FEATURES_CONTAINER)
        for blob in container.list_blobs():
            name = blob.name  # e.g. 'EURUSD.sim_features.csv'
            if name.endswith('_features.csv'):
                sym = name[:-len('_features.csv')]  # e.g. 'EURUSD.sim'
                live_symbols.add(sym)
        print(f'Live feature symbols in Azure: {sorted(live_symbols)}')
    except Exception as e:
        print(f'Could not list Azure features container: {e}')
        # Fallback: build from model bases assuming .sim suffix
        live_symbols = {f'{b}.sim' for b in model_bases}

    # Step 3: build map — strip .sim suffix to match model base names
    symbol_map = {}
    for sym_full in sorted(live_symbols):
        # Strip known broker suffixes to get base name
        base = sym_full
        for sfx in ('.sim', '.i', '_SB', '.r', '.a', '.b', '.m', '.pro'):
            if base.endswith(sfx):
                base = base[:-len(sfx)]
                break
        base = base.upper()
        if base in model_bases:
            symbol_map[sym_full] = base

    return symbol_map


SYMBOL_MAP = discover_symbol_map()
print(f'\nActive inference symbols ({len(SYMBOL_MAP)}):')
for full, base in sorted(SYMBOL_MAP.items()):
    lgbm_ok = base in lgbm_models
    cb_ok   = base in catboost_models
    print(f'  {full:<20} LGB={lgbm_ok}  CAT={cb_ok}')

In [ ]:
# ── Cell 9: MAIN INFERENCE LOOP — runs forever ───────────────────────────
# Leave this cell running. It polls Azure every 30 seconds,
# runs LGB + CatBoost + Transformer inference on all available symbols,
# and uploads non-HOLD predictions so live_trading_system.py can consume them.

import time
from IPython.display import clear_output

POLL_INTERVAL = 30   # seconds between inference runs
cycle = 0

n_lgb = len(lgbm_models)
n_cat = len(catboost_models)
n_trf = len(transformer_models)

print('=== FTMO AI Trading System — Colab Inference ===')
print(f'Models: {n_lgb} LGB  +  {n_cat} CatBoost  +  {n_trf} Transformer')
print(f'Symbols: {list(SYMBOL_MAP.keys())}')
print(f'Poll interval: {POLL_INTERVAL}s')
print('Running... (interrupt kernel to stop)')
print()

while True:
    cycle += 1
    ts = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    results_log = []

    for sym_full, base in SYMBOL_MAP.items():
        lgbm_r, cb_r, trf_r = run_inference_for_symbol(sym_full, base)

        def fmt(r):
            if r is None:  return '—'
            return f'{r[0]} ({r[1]:.2f})'

        results_log.append(
            f'  {base:<8}  LGB: {fmt(lgbm_r):<20}  CAT: {fmt(cb_r):<20}  TRF: {fmt(trf_r)}'
        )

    clear_output(wait=True)
    print(f'=== Cycle {cycle} · {ts} ===')
    for line in results_log:
        print(line)
    print(f'\nNext run in {POLL_INTERVAL}s...')

    try:
        heartbeat = {'timestamp': datetime.now(timezone.utc).isoformat(), 'cycle': cycle}
        blob_service.get_blob_client(container=PREDICTIONS_CONTAINER, blob='_heartbeat.json')\n            .upload_blob(json.dumps(heartbeat), overwrite=True)
    except Exception as e:
        print(f'heartbeat upload failed: {e}')

    time.sleep(POLL_INTERVAL)